In [1]:
import torch

print(torch.backends.mps.is_available())
print(torch.device("mps"))

True
mps


In [2]:
# Use %pip so packages install into the *same* Python as this notebook kernel
# (shell `pip3` often points at a different interpreter than the kernel).
%pip install -q rank_bm25 transformers datasets scikit-learn tqdm torch numpy pandas scipy accelerate sentence-transformers


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import json

with open("data/train-claims.json") as f:
    train_data = json.load(f)

with open("data/dev-claims.json") as f:
    dev_data = json.load(f)

with open("data/evidence.json") as f:
    evidence_data = json.load(f)

In [4]:
print(len(train_data))
print(len(evidence_data))

1228
1208827


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
from tqdm import tqdm

# evidence (shared by BM25 in the next cell)
eid_list = list(evidence_data.keys())
corpus = list(evidence_data.values())

# --- TF-IDF (disabled; use BM25 in the next cell) ---
# 构建 TF-IDF 稀疏矩阵
_tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    max_features=300000,
    ngram_range=(1, 2)
)

_tfidf_evidence_matrix = _tfidf_vectorizer.fit_transform(corpus)

# 检索函数：返回 TF-IDF 分数最高的 top-k evidence ids（默认 50，供 cross-encoder 再排序）
def _retrieve_tfidf_topk(claim, k=50):
    n = len(eid_list)
    k = min(k, n)
    if k == 0:
        return []
    query_vec = _tfidf_vectorizer.transform([claim])
    scores = (query_vec @ _tfidf_evidence_matrix.T).toarray()[0]
    top_k_idx = np.argpartition(scores, -k)[-k:]
    top_k_idx = top_k_idx[np.argsort(scores[top_k_idx])[::-1]]
    return [eid_list[i] for i in top_k_idx]


In [6]:
import re
import numpy as np
from tqdm import tqdm
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS


def _tokenize_bm25(text):
    text = (text or "").lower()
    tokens = re.findall(r"[a-z0-9]+", text)
    stop = ENGLISH_STOP_WORDS
    return [t for t in tokens if t not in stop]


# BM25 index (same role as the commented TF-IDF block above)
tokenized_corpus = [_tokenize_bm25(doc) for doc in tqdm(corpus, desc="Tokenize for BM25")]
tokenized_corpus = [t if t else ["_"] for t in tokenized_corpus]
bm25 = BM25Okapi(tokenized_corpus)


# BM25 检索（命名与诊断区 `_retrieve_tfidf_topk` 对齐）
def _retrieve_bm25_topk(claim, k=50):
    n = len(eid_list)
    k = min(k, n)
    if k == 0:
        return []
    q_tokens = _tokenize_bm25(claim)
    if not q_tokens:
        q_tokens = ["_"]
    scores = bm25.get_scores(q_tokens)
    top_k_idx = np.argpartition(scores, -k)[-k:]
    top_k_idx = top_k_idx[np.argsort(scores[top_k_idx])[::-1]]
    return [eid_list[i] for i in top_k_idx]


retrieve_top_k_fast = _retrieve_bm25_topk


Tokenize for BM25: 100%|██████████| 1208827/1208827 [00:05<00:00, 233555.60it/s]


### 检索与超参诊断区（非正式训练）

从 **TF-IDF / BM25 vs gold** 到 **并集 vs gold** 为止，仅用于检索与召回的可视化调参；其下 **Cross-encoder 微调** 后，才是正式 **Cross-encoder 重排** 与 `build_dataset` / Trainer。须已执行 **TF-IDF gold** 与 **BM25 检索**。


### Gold evidences vs **TF-IDF** top-50（dev，`dev-claims.json`）

在 **`dev_data`** 上统计（不要用 `train_data`）。单独拟合 TF-IDF 仅用于看 gold 命中率；主检索路径为 BM25（`_retrieve_bm25_topk`，别名 `retrieve_top_k_fast`）。运行下方 cell 后得到 **`tf_idf_top50`**：`claim_id -> top-50 evidence ids`（仅含至少一条 gold 的 claim）。


In [10]:
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# TF-IDF built only for this diagnostic (same hyperparameters as the commented retrieval cell).
_tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    max_features=300000,
    ngram_range=(1, 2),
)
_tfidf_evidence_matrix = _tfidf_vectorizer.fit_transform(corpus)


def _retrieve_tfidf_topk(claim, k=50):
    n = len(eid_list)
    k = min(k, n)
    if k == 0:
        return []
    query_vec = _tfidf_vectorizer.transform([claim])
    scores = (query_vec @ _tfidf_evidence_matrix.T).toarray()[0]
    top_k_idx = np.argpartition(scores, -k)[-k:]
    top_k_idx = top_k_idx[np.argsort(scores[top_k_idx])[::-1]]
    return [eid_list[i] for i in top_k_idx]


tf_idf_top50 = {}


def tfidf_topk_gold_stats(claims_data, k=50):
    """Gold evidence IDs found in TF-IDF top-k (per claim and pooled)."""
    tf_idf_top50.clear()
    total_gold = 0
    total_hits = 0
    claims_with_any_hit = 0
    hits_per_claim = []

    for cid, item in tqdm(claims_data.items(), desc=f"TF-IDF top-{k} vs gold"):
        claim = item["claim_text"]
        gold = item.get("evidences") or []
        gold_set = set(gold)
        if not gold_set:
            continue
        top_eids = _retrieve_tfidf_topk(claim, k=k)
        tf_idf_top50[cid] = top_eids
        top_set = set(top_eids)
        n_hit = len(gold_set & top_set)
        total_gold += len(gold_set)
        total_hits += n_hit
        hits_per_claim.append(n_hit)
        if n_hit > 0:
            claims_with_any_hit += 1

    n = len(hits_per_claim)
    if n == 0:
        print("No claims with non-empty gold evidences.")
        return
    mean_hits = sum(hits_per_claim) / n
    recall_at_k = total_hits / total_gold if total_gold else 0.0
    print(f"Claims with gold evidences: {n}")
    print(f"Total gold evidence links: {total_gold}")
    print(f"Gold links that appear in TF-IDF top-{k}: {total_hits}")
    print(f"Mean gold hits per claim in top-{k}: {mean_hits:.3f}")
    print(f"Recall@{k} (hits / all gold links): {recall_at_k:.3f}")
    print(f"Share of claims with at least one gold in top-{k}: {claims_with_any_hit / n:.3f}")


tfidf_topk_gold_stats(dev_data, k=50)


TF-IDF top-50 vs gold: 100%|██████████| 154/154 [00:12<00:00, 12.16it/s]

Claims with gold evidences: 154
Total gold evidence links: 491
Gold links that appear in TF-IDF top-50: 129
Mean gold hits per claim in top-50: 0.838
Recall@50 (hits / all gold links): 0.263
Share of claims with at least one gold in top-50: 0.532


### Gold evidences vs **BM25** top-50（dev，`dev-claims.json`）

在 **`dev_data`** 上统计（不要用 `train_data`）；使用上方 BM25 检索单元里的 **`_retrieve_bm25_topk`**（与 `retrieve_top_k_fast` 相同）。运行下方 cell 后得到 **`bm25_top50`**（`claim_id -> top-50 evidence ids`，仅含至少一条 gold 的 claim）。


In [11]:
from tqdm import tqdm


bm25_top50 = {}


def bm25_topk_gold_stats(claims_data, k=50):
    """Gold evidence IDs found in BM25 top-k (uses _retrieve_bm25_topk from the BM25 cell)."""
    bm25_top50.clear()
    total_gold = 0
    total_hits = 0
    claims_with_any_hit = 0
    hits_per_claim = []

    for cid, item in tqdm(claims_data.items(), desc=f"BM25 top-{k} vs gold"):
        claim = item["claim_text"]
        gold = item.get("evidences") or []
        gold_set = set(gold)
        if not gold_set:
            continue
        top_eids = _retrieve_bm25_topk(claim, k=k)
        bm25_top50[cid] = top_eids
        top_set = set(top_eids)
        n_hit = len(gold_set & top_set)
        total_gold += len(gold_set)
        total_hits += n_hit
        hits_per_claim.append(n_hit)
        if n_hit > 0:
            claims_with_any_hit += 1

    n = len(hits_per_claim)
    if n == 0:
        print("No claims with non-empty gold evidences.")
        return
    mean_hits = sum(hits_per_claim) / n
    recall_at_k = total_hits / total_gold if total_gold else 0.0
    print(f"Claims with gold evidences: {n}")
    print(f"Total gold evidence links: {total_gold}")
    print(f"Gold links that appear in BM25 top-{k}: {total_hits}")
    print(f"Mean gold hits per claim in top-{k}: {mean_hits:.3f}")
    print(f"Recall@{k} (hits / all gold links): {recall_at_k:.3f}")
    print(f"Share of claims with at least one gold in top-{k}: {claims_with_any_hit / n:.3f}")


bm25_topk_gold_stats(dev_data, k=50)


BM25 top-50 vs gold: 100%|██████████| 154/154 [02:45<00:00,  1.07s/it]

Claims with gold evidences: 154
Total gold evidence links: 491
Gold links that appear in BM25 top-50: 186
Mean gold hits per claim in top-50: 1.208
Recall@50 (hits / all gold links): 0.379
Share of claims with at least one gold in top-50: 0.682


### BM25 top-50 ∪ TF-IDF top-50 vs gold（dev，`dev-claims.json`）

本 cell **不再次检索**：仅读取 **`tf_idf_top50`** 与 **`bm25_top50`**，用 **`union_topk_vs_gold_sets`** / **`print_union_cached_vs_gold`** 统计并集与 gold。须先依次运行 TF-IDF gold 单元、BM25 gold 单元。


In [10]:
from tqdm import tqdm

if "tf_idf_top50" not in globals() or "bm25_top50" not in globals():
    raise RuntimeError(
        "请先依次运行 TF-IDF gold 单元与 BM25 gold 单元，以生成 tf_idf_top50 与 bm25_top50。"
    )


def union_topk_candidate_eids(bm_eids, tf_eids):
    """BM25 在前、再补 TF-IDF 独有 id（与正式 retrieve_reranked 一致）。"""
    return list(dict.fromkeys(list(bm_eids) + list(tf_eids)))


def union_topk_vs_gold_sets(bm_eids, tf_eids):
    """BM25 与 TF-IDF 两段 top-k id 列表 → (T, B, U) 三个集合，用于与 gold 做集合统计。"""
    T = set(tf_eids)
    B = set(bm_eids)
    return T, B, T | B


def print_union_cached_vs_gold(dev_data, tf_idf_top50, bm25_top50):
    sg = su = sb = st = sm = sn = 0
    gold_ranks = []
    for cid, item in tqdm(dev_data.items(), desc="union-vs-gold"):
        G = set(item.get("evidences") or [])
        if not G:
            continue
        bm_eids = bm25_top50.get(cid, ())
        tf_eids = tf_idf_top50.get(cid, ())
        T, B, U = union_topk_vs_gold_sets(bm_eids, tf_eids)
        U_ordered = union_topk_candidate_eids(bm_eids, tf_eids)
        rank_in_u = {eid: i + 1 for i, eid in enumerate(U_ordered)}
        for eid in G:
            if eid in rank_in_u:
                gold_ranks.append(rank_in_u[eid])
        sg += len(G)
        su += len(G & U)
        sb += len(G & T & B)
        st += len(G & (T - B))
        sm += len(G & (B - T))
        sn += len(G - U)

    print("=== BM25 ∪ TF-IDF top-50 vs gold (dev_data, cached) ===")
    print("Total gold evidence links:", sg)
    print("Gold in union U:", su)
    print("  Both BM25 and TF-IDF:", sb)
    print("  TF-IDF only (in T not B):", st)
    print("  BM25 only (in B not T):", sm)
    print("Gold not in union:", sn)
    if sg:
        print(f"Union recall |G∩U|/|G|: {su / sg:.4f}")
        print(f"Both / |G|: {sb / sg:.4f}")
        print(f"TF-IDF only / |G|: {st / sg:.4f}")
        print(f"BM25 only / |G|: {sm / sg:.4f}")
        print(f"Outside union / |G|: {sn / sg:.4f}")
    print(
        "Partition check (both+tfidf_only+bm25_only+outside == total gold links):",
        sb + st + sm + sn == sg,
    )
    if gold_ranks:
        mean_gold_rank_in_union = sum(gold_ranks) / len(gold_ranks)
        print(
            f"mean_gold_rank_in_union: {mean_gold_rank_in_union:.3f} "
            f"(over {len(gold_ranks)} gold links in U, 1-based rank in BM25→TF-IDF list)"
        )
    else:
        print("mean_gold_rank_in_union: n/a (no gold in union)")


print_union_cached_vs_gold(dev_data, tf_idf_top50, bm25_top50)


union-vs-gold: 100%|██████████| 154/154 [00:00<00:00, 4865.27it/s]

=== BM25 ∪ TF-IDF top-50 vs gold (dev_data, cached) ===
Total gold evidence links: 491
Gold in union U: 195
  Both BM25 and TF-IDF: 120
  TF-IDF only (in T not B): 9
  BM25 only (in B not T): 66
Gold not in union: 296
Union recall |G∩U|/|G|: 0.3971
Both / |G|: 0.2444
TF-IDF only / |G|: 0.0183
BM25 only / |G|: 0.1344
Outside union / |G|: 0.6029
Partition check (both+tfidf_only+bm25_only+outside == total gold links): True
mean_gold_rank_in_union: 15.862 (over 195 gold links in U, 1-based rank in BM25→TF-IDF list)


### Cross-encoder 微调（`train_data`，union 内 gold / non-gold）

在正式 `retrieve_reranked` 之前，用 **train** 上 BM25∪TF-IDF 的并集 `U` 构造二分类数据：
- **正例**：`e ∈ gold ∩ U`，`(claim, evidence)` → label `1`
- **负例**：`e ∈ U \ gold`，用**未微调**的 CE 在 non-gold 上取 top 难负例 + 少量随机负例 → label `0`

顺序：① 建 `train_union_cache` → ② `ce_train_examples` → ③ `fit` 后 **`cross_encoder.save`** 到项目根目录 **`ce_finetuned/`**。再运行下方正式 Cross-encoder cell。


In [ ]:
from tqdm import tqdm

RETRIEVE_K_CE = 50


def union_topk_candidate_eids(bm_eids, tf_eids):
    """BM25 在前、再补 TF-IDF 独有 id（与正式管线一致）。"""
    return list(dict.fromkeys(list(bm_eids) + list(tf_eids)))


if "_retrieve_tfidf_topk" not in globals():
    raise RuntimeError("请先运行 TF-IDF gold 单元（定义 _retrieve_tfidf_topk）。")
if "_retrieve_bm25_topk" not in globals():
    raise RuntimeError("请先运行 BM25 检索单元（定义 _retrieve_bm25_topk）。")

train_union_cache = {}
n_claims = n_gold_links = n_gold_in_u = n_non_gold = 0

for cid, item in tqdm(train_data.items(), desc="train: U, gold, U\\gold"):
    gold = set(item.get("evidences") or [])
    if not gold:
        continue
    claim = item["claim_text"]
    bm = _retrieve_bm25_topk(claim, k=RETRIEVE_K_CE)
    tf = _retrieve_tfidf_topk(claim, k=RETRIEVE_K_CE)
    U = union_topk_candidate_eids(bm, tf)
    gold_in_U = [e for e in U if e in gold]
    non_gold = [e for e in U if e not in gold]
    train_union_cache[cid] = {
        "claim": claim,
        "gold": gold,
        "U": U,
        "gold_in_U": gold_in_U,
        "non_gold": non_gold,
    }
    n_claims += 1
    n_gold_links += len(gold)
    n_gold_in_u += len(gold_in_U)
    n_non_gold += len(non_gold)

print(f"Train claims with gold: {n_claims}")
print(f"Total gold evidence links: {n_gold_links}")
print(f"Gold in union |G∩U|: {n_gold_in_u}  (recall vs all gold links: {n_gold_in_u / n_gold_links:.4f})")
print(f"Non-gold slots in U (sum over claims): {n_non_gold}")


train: U, gold, U\gold:   0%|          | 4/1228 [00:06<31:42,  1.55s/it]


KeyboardInterrupt: 

In [ ]:
import random

import numpy as np
import torch
from sentence_transformers import CrossEncoder, InputExample
from tqdm import tqdm

NUM_HARD_NEG = 8
NUM_RANDOM_NEG = 2
CE_MINE_BATCH = 32
CROSS_ENCODER_BASE = "cross-encoder/ms-marco-MiniLM-L-6-v2"

if "train_union_cache" not in globals():
    raise RuntimeError("请先运行上一 cell，生成 train_union_cache。")

_ce_mine_kw = {}
if torch.cuda.is_available():
    _ce_mine_kw["device"] = "cuda"
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    _ce_mine_kw["device"] = "mps"
ce_miner = CrossEncoder(CROSS_ENCODER_BASE, **_ce_mine_kw)
print("Hard-negative miner device:", _ce_mine_kw.get("device", "cpu (default)"))

rng = random.Random(42)


def pick_hard_negatives(claim, non_gold_eids, k):
    if not non_gold_eids or k <= 0:
        return []
    k = min(k, len(non_gold_eids))
    pairs = [(claim, evidence_data[eid]) for eid in non_gold_eids]
    scores = ce_miner.predict(pairs, show_progress_bar=False, batch_size=CE_MINE_BATCH)
    order = np.argsort(scores)[::-1][:k]
    return [non_gold_eids[i] for i in order]


ce_train_examples = []
n_pos = n_neg = 0

for cid, rec in tqdm(train_union_cache.items(), desc="build CE train pairs"):
    claim = rec["claim"]
    for eid in rec["gold_in_U"]:
        ce_train_examples.append(
            InputExample(texts=[claim, evidence_data[eid]], label=1.0)
        )
        n_pos += 1
    non_gold = rec["non_gold"]
    if not non_gold:
        continue
    hard = pick_hard_negatives(claim, non_gold, NUM_HARD_NEG)
    hard_set = set(hard)
    pool = [e for e in non_gold if e not in hard_set]
    rand_k = min(NUM_RANDOM_NEG, len(pool))
    easy = rng.sample(pool, rand_k) if rand_k else []
    for eid in hard + easy:
        ce_train_examples.append(
            InputExample(texts=[claim, evidence_data[eid]], label=0.0)
        )
        n_neg += 1

print(f"Examples: {len(ce_train_examples)}  (positive={n_pos}, negative={n_neg})")


In [ ]:
from pathlib import Path

import torch
from torch.utils.data import DataLoader
from sentence_transformers import CrossEncoder

if "ce_train_examples" not in globals() or not ce_train_examples:
    raise RuntimeError("请先运行构造数据集 cell（ce_train_examples 为空）。")

CROSS_ENCODER_BASE = globals().get(
    "CROSS_ENCODER_BASE", "cross-encoder/ms-marco-MiniLM-L-6-v2"
)
CE_FINETUNE_DIR = Path("ce_finetuned").resolve()
CE_EPOCHS = 1
CE_BATCH_SIZE = 16
CE_LR = 2e-5

ce_train_dataloader = DataLoader(
    ce_train_examples, shuffle=True, batch_size=CE_BATCH_SIZE
)

cross_encoder = CrossEncoder(
    CROSS_ENCODER_BASE,
    num_labels=1,
    **_ce_mine_kw,
)

# 训练前快照：用于确认参数是否真的更新（v4 的 fit() 在部分环境下 save 仍是基座权重）
_param_snap = {
    n: p.detach().cpu().clone()
    for n, p in cross_encoder.model.named_parameters()
}

warmup = min(100, max(1, len(ce_train_examples) // CE_BATCH_SIZE))
# 使用 old_fit：直接在 self.model 上反传，避免 v4 CrossEncoderTrainer 训完但权重未写入 save 的问题
cross_encoder.old_fit(
    train_dataloader=ce_train_dataloader,
    epochs=CE_EPOCHS,
    warmup_steps=warmup,
    optimizer_params={"lr": CE_LR},
    show_progress_bar=True,
)

_max_delta = max(
    (p.detach().cpu() - _param_snap[n]).abs().max().item()
    for n, p in cross_encoder.model.named_parameters()
)
print(f"Max |param_after - param_before|: {_max_delta:.6e}")
if _max_delta < 1e-7:
    raise RuntimeError("训练后参数几乎未变，请勿 save；检查 device / loss / dataloader。")

CE_FINETUNE_DIR.mkdir(parents=True, exist_ok=True)
cross_encoder.save(str(CE_FINETUNE_DIR))
print("cwd:", Path.cwd())
print("Fine-tuned CrossEncoder saved to:", CE_FINETUNE_DIR)
print("Files:", sorted(p.name for p in CE_FINETUNE_DIR.iterdir())[:8], "...")


In [ ]:
# 仅当上一 cell 已 old_fit 且 max param delta > 0 时再运行；勿对未训练的 cross_encoder save（会覆盖成基座）
from pathlib import Path

CE_FINETUNE_DIR = Path("ce_finetuned").resolve()
CE_FINETUNE_DIR.mkdir(parents=True, exist_ok=True)
cross_encoder.save(str(CE_FINETUNE_DIR))
print(CE_FINETUNE_DIR, list(CE_FINETUNE_DIR.iterdir())[:5])

### Cross-encoder 微调前后 vs gold（dev，BM25∪TF-IDF 并集内重排）

在同一候选集 `U`（BM25∪TF-IDF top-50）上重排，重点看 **CE top-5 找回了 union 里多少 gold**。须已运行 dev 的 TF-IDF/BM25 gold 单元且存在 `ce_finetuned/`。


In [ ]:
from pathlib import Path

import numpy as np
import torch
from sentence_transformers import CrossEncoder
from tqdm import tqdm

CE_COMPARE_RETRIEVE_K = 50
CE_COMPARE_TOPK = 5
CE_COMPARE_BATCH = 32
CROSS_ENCODER_BASE = "cross-encoder/ms-marco-MiniLM-L-6-v2"
CE_FINETUNE_DIR = Path("ce_finetuned").resolve()

if "tf_idf_top50" not in globals() or "bm25_top50" not in globals():
    raise RuntimeError("请先运行 dev 上 TF-IDF / BM25 gold 单元，生成 tf_idf_top50 与 bm25_top50。")
if not (CE_FINETUNE_DIR / "config.json").is_file():
    raise RuntimeError(f"未找到微调权重: {CE_FINETUNE_DIR}，请先运行 CE 微调并 save。")

_ce_dev_kw = {}
if torch.cuda.is_available():
    _ce_dev_kw["device"] = "cuda"
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    _ce_dev_kw["device"] = "mps"

ce_base = CrossEncoder(CROSS_ENCODER_BASE, **_ce_dev_kw)
ce_finetuned = CrossEncoder(str(CE_FINETUNE_DIR), **_ce_dev_kw)
print("Compare on dev_data | device:", _ce_dev_kw.get("device", "cpu"))


def _union_from_cache(cid):
    bm = bm25_top50.get(cid, ())
    tf = tf_idf_top50.get(cid, ())
    if bm or tf:
        return list(dict.fromkeys(list(bm) + list(tf)))
    return []


def _ce_rank_eids(model, claim, cand_eids):
    if not cand_eids:
        return []
    pairs = [(claim, evidence_data[eid]) for eid in cand_eids]
    scores = model.predict(pairs, show_progress_bar=False, batch_size=CE_COMPARE_BATCH)
    order = np.argsort(scores)[::-1]
    return [cand_eids[i] for i in order]


def _eval_ce_gold(model, name):
    total_gold_links = 0
    hits_topk = 0
    total_gold_in_u = 0
    hits_gold_in_u_topk = 0
    claims_with_gold = 0
    claims_any_hit_topk = 0
    claims_any_hit_gold_in_u_topk = 0
    mrr_list = []
    gold_ranks = []

    for cid, item in tqdm(dev_data.items(), desc=f"CE {name}"):
        gold = set(item.get("evidences") or [])
        if not gold:
            continue
        claim = item["claim_text"]
        U = _union_from_cache(cid)
        if not U:
            bm = _retrieve_bm25_topk(claim, k=CE_COMPARE_RETRIEVE_K)
            tf = _retrieve_tfidf_topk(claim, k=CE_COMPARE_RETRIEVE_K)
            U = list(dict.fromkeys(list(bm) + list(tf)))
        if not U:
            continue

        ranked = _ce_rank_eids(model, claim, U)
        top_set = set(ranked[:CE_COMPARE_TOPK])
        gold_in_u = gold & set(U)

        total_gold_links += len(gold)
        hits_topk += len(gold & top_set)
        total_gold_in_u += len(gold_in_u)
        hits_gold_in_u_topk += len(gold_in_u & top_set)
        claims_with_gold += 1
        if gold & top_set:
            claims_any_hit_topk += 1
        if gold_in_u & top_set:
            claims_any_hit_gold_in_u_topk += 1

        best_rank = None
        for g in gold_in_u:
            r = ranked.index(g) + 1
            gold_ranks.append(r)
            if best_rank is None or r < best_rank:
                best_rank = r
        if best_rank is not None:
            mrr_list.append(1.0 / best_rank)

    recall_topk = hits_topk / total_gold_links if total_gold_links else 0.0
    recall_gold_in_u_topk = (
        hits_gold_in_u_topk / total_gold_in_u if total_gold_in_u else 0.0
    )
    claim_hit_rate = claims_any_hit_topk / claims_with_gold if claims_with_gold else 0.0
    claim_hit_gold_in_u = (
        claims_any_hit_gold_in_u_topk / claims_with_gold if claims_with_gold else 0.0
    )
    mrr = float(np.mean(mrr_list)) if mrr_list else 0.0
    mean_rank = float(np.mean(gold_ranks)) if gold_ranks else float("nan")
    return {
        "name": name,
        "claims": claims_with_gold,
        "gold_in_U": total_gold_in_u,
        f"gold_in_U_hit_top{CE_COMPARE_TOPK}": hits_gold_in_u_topk,
        f"recall@{CE_COMPARE_TOPK}_gold_in_U": recall_gold_in_u_topk,
        "total_gold_links": total_gold_links,
        f"gold_hit_top{CE_COMPARE_TOPK}": hits_topk,
        f"recall@{CE_COMPARE_TOPK}_all_gold": recall_topk,
        "claim_hit_rate": claim_hit_rate,
        "claim_hit_gold_in_U": claim_hit_gold_in_u,
        "mrr": mrr,
        "mean_gold_rank_in_U": mean_rank,
        "gold_in_U_ranks_n": len(gold_ranks),
    }


stats_base = _eval_ce_gold(ce_base, "base")
stats_ft = _eval_ce_gold(ce_finetuned, "finetuned")

print("=== CE top-5 vs gold（dev，候选 U = BM25∪TF-IDF top-50）===")
print(f"CE 重排后取 top-{CE_COMPARE_TOPK}")
print()
print("【核心】top-50 并集 U 里的 gold，有多少被 CE top-5 找回：|G∩U∩top5| / |G∩U|")
for label, st in [("base", stats_base), ("finetuned", stats_ft)]:
    hit = st[f"gold_in_U_hit_top{CE_COMPARE_TOPK}"]
    tot = st["gold_in_U"]
    rec = st[f"recall@{CE_COMPARE_TOPK}_gold_in_U"]
    print(f"  {label:10s}  {hit}/{tot} gold-in-U links in top-{CE_COMPARE_TOPK}  recall={rec:.4f}")
b, f = stats_base[f"recall@{CE_COMPARE_TOPK}_gold_in_U"], stats_ft[f"recall@{CE_COMPARE_TOPK}_gold_in_U"]
print(f"  delta (finetuned-base) recall@5 on gold-in-U: {f - b:+.4f}")
print()
print("【参考】全部 gold（含不在 U 里的）被 top-5 找回：|G∩top5| / |G|")
for label, st in [("base", stats_base), ("finetuned", stats_ft)]:
    hit = st[f"gold_hit_top{CE_COMPARE_TOPK}"]
    tot = st["total_gold_links"]
    rec = st[f"recall@{CE_COMPARE_TOPK}_all_gold"]
    print(f"  {label:10s}  {hit}/{tot}  recall={rec:.4f}")
print()
for key in ("claim_hit_gold_in_U", "claim_hit_rate", "mrr", "mean_gold_rank_in_U"):
    b, f = stats_base[key], stats_ft[key]
    delta = (b - f) if key == "mean_gold_rank_in_U" else (f - b)
    sign = "+" if delta >= 0 else ""
    print(f"{key:28s}  base={b:.4f}  finetuned={f:.4f}  ({sign}{delta:.4f})")


### CE top-5 证据分数明细（dev，154 claims）

对每条有 gold 的 dev claim：在 **U = BM25∪TF-IDF top-50** 上用 **微调后 CE** 打分，输出 **top-5** 明细到 `ce_top5_scores_dev.json`。


In [ ]:
import json
from pathlib import Path

import numpy as np
import torch
from sentence_transformers import CrossEncoder
from tqdm import tqdm

CE_SCORE_RETRIEVE_K = 50
CE_SCORE_TOPK = 5
CE_SCORE_BATCH = 32
CE_FINETUNE_DIR = Path("ce_finetuned").resolve()
CROSS_ENCODER_BASE = globals().get(
    "CROSS_ENCODER_BASE", "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

if "tf_idf_top50" not in globals() or "bm25_top50" not in globals():
    raise RuntimeError("请先运行 dev TF-IDF / BM25 gold 单元。")
if not (CE_FINETUNE_DIR / "config.json").is_file():
    raise RuntimeError(f"缺少 {CE_FINETUNE_DIR}，请先 CE 微调并 save。")

if "ce_finetuned" not in globals():
    _kw = {}
    if torch.cuda.is_available():
        _kw["device"] = "cuda"
    elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        _kw["device"] = "mps"
    ce_finetuned = CrossEncoder(str(CE_FINETUNE_DIR), **_kw)


def _union_for_claim(cid, claim):
    bm = bm25_top50.get(cid, ())
    tf = tf_idf_top50.get(cid, ())
    if bm or tf:
        return list(dict.fromkeys(list(bm) + list(tf)))
    bm = _retrieve_bm25_topk(claim, k=CE_SCORE_RETRIEVE_K)
    tf = _retrieve_tfidf_topk(claim, k=CE_SCORE_RETRIEVE_K)
    return list(dict.fromkeys(list(bm) + list(tf)))


def _ce_topk_scored(model, claim, cand_eids, topk=CE_SCORE_TOPK):
    if not cand_eids:
        return []
    pairs = [(claim, evidence_data[eid]) for eid in cand_eids]
    scores = model.predict(pairs, show_progress_bar=False, batch_size=CE_SCORE_BATCH)
    order = np.argsort(scores)[::-1][: min(topk, len(cand_eids))]
    return [(cand_eids[i], float(scores[i])) for i in order]


ce_top5_scores_dev = {}

for cid, item in tqdm(dev_data.items(), desc="CE top-5 scores"):
    gold = set(item.get("evidences") or [])
    if not gold:
        continue
    claim = item["claim_text"]
    U = _union_for_claim(cid, claim)
    if not U:
        continue
    top5 = _ce_topk_scored(ce_finetuned, claim, U)
    ce_top5_scores_dev[cid] = {
        "claim_text": claim,
        "gold_evidences": sorted(gold),
        "top5": [
            {"rank": r, "evidence_id": eid, "score": sc, "is_gold": eid in gold}
            for r, (eid, sc) in enumerate(top5, start=1)
        ],
    }

out_path = Path("ce_top5_scores_dev.json")
with open(out_path, "w") as f:
    json.dump(ce_top5_scores_dev, f, indent=2)

print(f"Claims with gold (scored): {len(ce_top5_scores_dev)}")
print(f"Saved -> {out_path.resolve()}")
print()

for cid in sorted(ce_top5_scores_dev.keys()):
    rec = ce_top5_scores_dev[cid]
    print(f"=== {cid} ===")
    print(rec["claim_text"][:120] + ("..." if len(rec["claim_text"]) > 120 else ""))
    for row in rec["top5"]:
        g = "GOLD" if row["is_gold"] else "    "
        print(f"  {row['rank']}. [{g}] {row['evidence_id']}  score={row['score']:.4f}")
    print()


### 正式训练（从 Cross-encoder 起）

以下 `retrieve_reranked`、`build_dataset`、`Trainer` 等为正式管线：每条 claim 先取 **BM25** 与 **TF-IDF** 各 `retrieve_k` 条证据 id，**并集去重**后交给 cross-encoder 重排到 `final_k`。

须已运行：**BM25 检索**、**TF-IDF gold**、以及上方 **CE 微调**（推荐；否则加载 `ms-marco` 基座）。


In [12]:
import os

import numpy as np
import torch
from sentence_transformers import CrossEncoder

from pathlib import Path

CROSS_ENCODER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
CE_FINETUNE_DIR = Path("ce_finetuned").resolve()

_ce_kw = {}
if torch.cuda.is_available():
    _ce_kw["device"] = "cuda"
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    _ce_kw["device"] = "mps"

_ce_has_finetuned = CE_FINETUNE_DIR.is_dir() and (CE_FINETUNE_DIR / "config.json").is_file()
_ce_load_path = str(CE_FINETUNE_DIR) if _ce_has_finetuned else CROSS_ENCODER_MODEL
print("cwd:", Path.cwd())
print("CrossEncoder load:", _ce_load_path, "(finetuned)" if _ce_has_finetuned else "(base)")
print("CrossEncoder device:", _ce_kw.get("device", "cpu (default)"))
cross_encoder = CrossEncoder(_ce_load_path, **_ce_kw)


def union_topk_candidate_eids(bm_eids, tf_eids):
    """BM25 ∪ TF-IDF 候选：先去重，顺序为 BM25 在前、再补 TF-IDF 独有。"""
    return list(dict.fromkeys(list(bm_eids) + list(tf_eids)))


def retrieve_reranked(claim, retrieve_k=50, final_k=5):
    """BM25 与 TF-IDF 各取 `retrieve_k` 条，合并去重后交 cross-encoder 重排到 `final_k`。"""
    if "_retrieve_tfidf_topk" not in globals():
        raise RuntimeError(
            "Union rerank 需要 `_retrieve_tfidf_topk`：请先运行 TF-IDF gold / 索引单元。"
        )
    bm_eids = _retrieve_bm25_topk(claim, k=retrieve_k)
    tf_eids = _retrieve_tfidf_topk(claim, k=retrieve_k)
    cand_eids = union_topk_candidate_eids(bm_eids, tf_eids)
    if not cand_eids:
        return []
    pairs = [(claim, evidence_data[eid]) for eid in cand_eids]
    scores = cross_encoder.predict(pairs, show_progress_bar=False)
    order = np.argsort(scores)[::-1][:final_k]
    return [cand_eids[i] for i in order]


import inspect
print("[OK] BM25+TF-IDF rerank:", inspect.signature(retrieve_reranked))


/opt/homebrew/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cwd: /Users/zhaowenji/Text-Classification
CrossEncoder load: /Users/zhaowenji/Text-Classification/ce_finetuned (finetuned)
CrossEncoder device: mps


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 9988.48it/s]


[OK] BM25+TF-IDF rerank: (claim, retrieve_k=50, final_k=5)


In [13]:
from tqdm import tqdm


def build_dataset(data, retrieve_k=50, final_k=5):
    texts = []
    labels = []

    label_map = {
        "SUPPORTS": 0,
        "REFUTES": 1,
        "NOT_ENOUGH_INFO": 2,
        "DISPUTED": 3,
    }

    for cid, item in tqdm(data.items()):
        claim = item["claim_text"]
        label = item["claim_label"]

        eids = retrieve_reranked(claim, retrieve_k=retrieve_k, final_k=final_k)

        evidences = [evidence_data[eid] for eid in eids]

        input_text = claim + " [SEP] " + " ".join(evidences)

        texts.append(input_text)
        labels.append(label_map[label])

    return texts, labels


In [14]:
train_texts, train_labels = build_dataset(train_data)
dev_texts, dev_labels = build_dataset(dev_data)

100%|██████████| 154/154 [03:16<00:00,  1.27s/it]


In [15]:
import transformers
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from transformers import Trainer, TrainingArguments
import torch
from torch.utils.data import Dataset
print(transformers.__file__)

/opt/homebrew/lib/python3.11/site-packages/transformers/__init__.py


In [16]:
class ClaimDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "labels": torch.tensor(self.labels[idx])
        }

In [17]:
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=4
)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 8691.06it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [18]:
train_dataset = ClaimDataset(train_texts, train_labels, tokenizer)
dev_dataset = ClaimDataset(dev_texts, dev_labels, tokenizer)

In [19]:
import gc
import itertools
import json
from pathlib import Path

import numpy as np
import torch
from sklearn.metrics import f1_score
from transformers import DistilBertForSequenceClassification, DistilBertTokenizer, Trainer, TrainingArguments

if "train_texts" not in globals() or "tokenizer" not in globals():
    raise RuntimeError("请先运行 build_dataset 与 DistilBERT tokenizer 单元。")
if "ClaimDataset" not in globals():
    raise RuntimeError("请先运行定义 ClaimDataset 的单元。")

# 网格：组合数 = len(LR) * len(EPOCHS) * len(MAX_LEN)（可自行增删）
# LEARNING_RATES = [2e-5, 3e-5]
# NUM_TRAIN_EPOCHS_LIST = [2, 3]
# MAX_LENGTHS = [256, 512]
LEARNING_RATES = [3e-5]
NUM_TRAIN_EPOCHS_LIST = [3]
MAX_LENGTHS = [512]

SWEEP_ROOT = Path("distilbert_hyperparam_sweep")
SWEEP_ROOT.mkdir(parents=True, exist_ok=True)
BACKBONE = "distilbert-base-uncased"


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": float((preds == labels).mean()),
        "macro_f1": float(f1_score(labels, preds, average="macro")),
    }


use_cuda = torch.cuda.is_available()
use_mps = bool(getattr(torch.backends, "mps", None) and torch.backends.mps.is_available())
print("Trainer accelerator:", "CUDA" if use_cuda else ("MPS" if use_mps else "CPU"))

sweep_results = []

for lr, epochs, max_len in itertools.product(
    LEARNING_RATES, NUM_TRAIN_EPOCHS_LIST, MAX_LENGTHS
):
    run_name = f"lr{lr}_ep{int(epochs)}_ml{int(max_len)}"
    run_dir = SWEEP_ROOT / run_name
    run_dir.mkdir(parents=True, exist_ok=True)

    print("\n" + "=" * 72)
    print(f"Run: {run_name}  ->  {run_dir}")
    print(f"  learning_rate={lr}  num_train_epochs={epochs}  max_length={max_len}")

    model = DistilBertForSequenceClassification.from_pretrained(
        BACKBONE, num_labels=4
    )
    train_ds = ClaimDataset(train_texts, train_labels, tokenizer, max_len=max_len)
    dev_ds = ClaimDataset(dev_texts, dev_labels, tokenizer, max_len=max_len)

    training_args = TrainingArguments(
        output_dir=str(run_dir / "trainer_output"),
        learning_rate=lr,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=epochs,
        eval_strategy="epoch",
        save_strategy="no",
        logging_steps=50,
        dataloader_pin_memory=use_cuda,
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=dev_ds,
        compute_metrics=compute_metrics,
    )
    trainer.train()
    metrics = trainer.evaluate()

    save_dir = run_dir / "saved_model"
    save_dir.mkdir(parents=True, exist_ok=True)
    trainer.save_model(str(save_dir))
    tokenizer.save_pretrained(str(save_dir))

    row = {
        "run_name": run_name,
        "save_dir": str(save_dir.resolve()),
        "learning_rate": lr,
        "num_train_epochs": epochs,
        "max_length": max_len,
    }
    row.update({k: float(v) for k, v in metrics.items()})
    sweep_results.append(row)

    with open(run_dir / "metrics.json", "w") as f:
        json.dump(row, f, indent=2)

    del trainer, model, train_ds, dev_ds
    gc.collect()
    if use_cuda:
        torch.cuda.empty_cache()
    elif use_mps:
        torch.mps.empty_cache()

with open(SWEEP_ROOT / "sweep_results.json", "w") as f:
    json.dump(sweep_results, f, indent=2)

best = max(sweep_results, key=lambda r: r.get("eval_macro_f1", 0.0))
print("\nBest by eval_macro_f1:", best["run_name"], "eval_macro_f1=", best.get("eval_macro_f1"))

model = DistilBertForSequenceClassification.from_pretrained(best["save_dir"])
tokenizer = DistilBertTokenizer.from_pretrained(best["save_dir"])
DISTILBERT_MAX_LEN_FOR_PREDICT = int(best["max_length"])
print("Loaded best run into `model` / `tokenizer` from:", best["save_dir"])
print("Set DISTILBERT_MAX_LEN_FOR_PREDICT =", DISTILBERT_MAX_LEN_FOR_PREDICT, "(for `predict`)")


Trainer accelerator: MPS

Run: lr3e-05_ep3_ml512  ->  distilbert_hyperparam_sweep/lr3e-05_ep3_ml512
  learning_rate=3e-05  num_train_epochs=3  max_length=512


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 11108.68it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,1.223415,1.269076,0.441558,0.153153
2,1.205889,1.224466,0.461039,0.259728
3,1.097387,1.227618,0.454545,0.294572


Training Loss,Validation Loss,Epoch,Accuracy,Macro F1
1.097387,1.227618,3,0.454545,0.294572


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  6.51it/s]



Best by eval_macro_f1: lr3e-05_ep3_ml512 eval_macro_f1= 0.2945717949428308


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 11703.99it/s]

Loaded best run into `model` / `tokenizer` from: /Users/zhaowenji/Text-Classification/distilbert_hyperparam_sweep/lr3e-05_ep3_ml512/saved_model
Set DISTILBERT_MAX_LEN_FOR_PREDICT = 512 (for `predict`)


In [20]:
import torch
import numpy as np
from tqdm import tqdm

use_cuda = torch.cuda.is_available()
use_mps = bool(getattr(torch.backends, "mps", None) and torch.backends.mps.is_available())
device = torch.device("cuda" if use_cuda else ("mps" if use_mps else "cpu"))
print("Predict device:", device)

model.to(device)
model.eval()

label_map_rev = {
    0: "SUPPORTS",
    1: "REFUTES",
    2: "NOT_ENOUGH_INFO",
    3: "DISPUTED",
}


def predict(data, retrieve_k=50, final_k=5):
    results = {}

    for cid, item in tqdm(data.items()):
        claim = item["claim_text"]

        eids = retrieve_reranked(claim, retrieve_k=retrieve_k, final_k=final_k)
        evidences = [evidence_data[eid] for eid in eids]

        input_text = claim + " [SEP] " + " ".join(evidences)

        max_len = int(globals().get("DISTILBERT_MAX_LEN_FOR_PREDICT", 512))
        inputs = tokenizer(
            input_text,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=max_len,
        )

        inputs = {key: value.to(device) for key, value in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)

        logits = outputs.logits.detach().float().cpu().numpy()
        pred = np.argmax(logits)

        results[cid] = {
            "claim_label": label_map_rev[pred],
            "evidences": eids,
        }

    return results


Predict device: mps


In [21]:
with open("data/test-claims-unlabelled.json") as f:
    test_data = json.load(f)
test_predictions = predict(test_data)
with open("test_predictions.json", "w") as f:
    json.dump(test_predictions, f, indent=2)

100%|██████████| 153/153 [03:23<00:00,  1.33s/it]


In [22]:
dev_predictions = predict(dev_data)
with open("dev_predictions.json", "w") as f:
    json.dump(dev_predictions, f, indent=2)

100%|██████████| 154/154 [03:27<00:00,  1.35s/it]


In [23]:
# 分类器
print("DistilBERT:", getattr(model, "name_or_path", model.config._name_or_path))

# 重排 CE
print("CrossEncoder:", cross_encoder.model.config.name_or_path if hasattr(cross_encoder, "model") else cross_encoder)

# predict 用的 max_len
print("max_len:", globals().get("DISTILBERT_MAX_LEN_FOR_PREDICT", "not set"))

DistilBERT: /Users/zhaowenji/Text-Classification/distilbert_hyperparam_sweep/lr3e-05_ep3_ml512/saved_model
CrossEncoder: /Users/zhaowenji/Text-Classification/ce_finetuned
max_len: 512


In [24]:
class Args:

  def __init__(self):

    self.predictions = "dev_predictions.json"
    # self.predictions = "test_predictions.json"

    # self.predictions = "data/dev-claims-baseline.json"

    self.groundtruth = "data/dev-claims.json"

    self.verbose = True

In [25]:
import argparse
import sys
import json
import numpy as np

######
#main#
######

def main(args):

    try:
        predictions = json.load(open(args.predictions))
    except:
        print("Error loading predictions json file:", args.predictions)
        raise SystemExit

    try:
        groundtruth = json.load(open(args.groundtruth))
    except:
        print("Error loading groundtruth json file:", args.groundtruth)
        raise SystemExit

    try:
        f, acc = [], []

        #iterate through the groundtruth instances
        for claim_id, claim in sorted(groundtruth.items()):
            if claim_id in predictions and \
                "claim_label" in predictions[claim_id] and \
                "evidences" in predictions[claim_id]:

                #check claim level label
                instance_correct = 0.0
                if predictions[claim_id]["claim_label"] == claim["claim_label"]:
                    instance_correct = 1.0

                #check retrieved evidences
                evidence_correct = 0
                evidence_recall = 0.0
                evidence_precision = 0.0
                evidence_fscore = 0.0
                if type(predictions[claim_id]["evidences"]) == list and (len(predictions[claim_id]["evidences"]) > 0):
                    top_six_ev = set(predictions[claim_id]["evidences"])
                    for gr_ev in claim["evidences"]:
                        if gr_ev in top_six_ev:
                            evidence_correct += 1
                    if evidence_correct > 0:
                        evidence_recall = float(evidence_correct) / len(claim["evidences"])
                        evidence_precision = \
                            float(evidence_correct) / len(predictions[claim_id]["evidences"])
                        evidence_fscore = (2*evidence_precision*evidence_recall)/(evidence_precision+evidence_recall)

                if args.verbose:
                    print("groundtruth =", claim)
                    print("predictions =", predictions[claim_id])
                    print("instance accuracy =", instance_correct)
                    print("evidence recall =", evidence_recall)
                    print("evidence precision =", evidence_precision)
                    print("evidence fscore =", evidence_fscore, "\n\n")

                #add the metric results
                acc.append(instance_correct)
                f.append(evidence_fscore)

        #compute aggregate performance
        mean_f = np.mean(f if len(f) > 0 else [0.0])
        mean_acc = np.mean(acc if len(acc) > 0 else [0.0])
        if mean_f == 0.0 and mean_acc == 0.0:
            hmean = 0.0
        else:
            hmean = (2*mean_f*mean_acc)/(mean_f+mean_acc)

        print("Evidence Retrieval F-score (F)    =", mean_f)
        print("Claim Classification Accuracy (A) =", mean_acc)
        print("Harmonic Mean of F and A          =", hmean)

    except Exception as error:
        print("Error:", error)
        raise SystemExit

if __name__ == "__main__":

    #parser arguments
    desc = "Evaluation script that computes evidence retrieval f-score, claim classification accuracy, and aggregate performance."
    parser = argparse.ArgumentParser(description=desc)

    #arguments
    # parser.add_argument("--predictions", required=True, help="json file containing the claim label predictions and retrieved evidences produced by a system")
    # parser.add_argument("--groundtruth", required=True, help="json file containing the ground truth claim labels and evidences")
    # parser.add_argument("--verbose", action="store_true", help="turn on debug prints")
    # args = parser.parse_args()

    args = Args()

    main(args)

groundtruth = {'claim_text': 'The corals may save themselves, as many other creatures are attempting to do, by moving toward the poles as the Earth warms, establishing new reefs in cooler water.”', 'claim_label': 'SUPPORTS', 'evidences': ['evidence-242575', 'evidence-1175280']}
predictions = {'claim_label': 'SUPPORTS', 'evidences': ['evidence-1175280', 'evidence-940910', 'evidence-528224', 'evidence-546845', 'evidence-1109959']}
instance accuracy = 1.0
evidence recall = 0.5
evidence precision = 0.2
evidence fscore = 0.28571428571428575 


groundtruth = {'claim_text': 'Increases in atmospheric CO2 followed increases in temperature.', 'claim_label': 'SUPPORTS', 'evidences': ['evidence-368192', 'evidence-423643', 'evidence-629358']}
predictions = {'claim_label': 'SUPPORTS', 'evidences': ['evidence-814004', 'evidence-498380', 'evidence-492188', 'evidence-364767', 'evidence-1065833']}
instance accuracy = 1.0
evidence recall = 0.0
evidence precision = 0.0
evidence fscore = 0.0 


groundtruth

# Retrieval:
1. TF-IDF top50
2. bm25 top50
3. union -> CE rerank top5
